# Gaussian Naive Bayes

Classificador probabilistico baseado em Bayes com a suposicao (ingenua)
de independencia entre features dada a classe.

Para cada classe, estimamos media e variancia de cada feature e usamos
a densidade normal como verossimilhanca. Trabalhamos em log-espaco para
evitar underflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
)

In [ ]:
class GaussianNaiveBayes:
    """
    Naive Bayes com verossimilhanca Gaussiana.

    Assume independencia condicional entre features dada a classe:
        P(y|x) propto P(y) * prod_j P(x_j | y)
    com cada P(x_j | y) modelado por uma normal univariada.

    Trabalhamos em log-espaco para estabilidade numerica.
    """

    def __init__(self, var_smoothing=1e-9):
        if var_smoothing < 0:
            raise ValueError("var_smoothing must be non-negative.")
        self.var_smoothing = var_smoothing

        self.classes_ = None
        self.class_prior_ = None
        self.theta_ = None  # media por classe
        self.var_ = None    # variancia por classe

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).ravel()

        if X.ndim != 2:
            raise ValueError("X must be 2D.")
        if len(X) != len(y):
            raise ValueError("X and y must have the same length.")

        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        self.theta_ = np.zeros((n_classes, n_features))
        self.var_ = np.zeros((n_classes, n_features))
        self.class_prior_ = np.zeros(n_classes)

        # smoothing baseado na variancia global (como sklearn faz)
        epsilon = self.var_smoothing * X.var(axis=0).max()

        for idx, cls in enumerate(self.classes_):
            X_cls = X[y == cls]
            self.theta_[idx] = X_cls.mean(axis=0)
            self.var_[idx] = X_cls.var(axis=0) + epsilon
            self.class_prior_[idx] = X_cls.shape[0] / X.shape[0]

        return self

    def _joint_log_likelihood(self, X):
        jll = np.zeros((X.shape[0], len(self.classes_)))
        for idx in range(len(self.classes_)):
            prior = np.log(self.class_prior_[idx])
            var = self.var_[idx]
            mean = self.theta_[idx]

            log_likelihood = -0.5 * np.sum(np.log(2.0 * np.pi * var))
            log_likelihood -= 0.5 * np.sum(((X - mean) ** 2) / var, axis=1)
            jll[:, idx] = prior + log_likelihood
        return jll

    def predict(self, X):
        if self.classes_ is None:
            raise ValueError("Call fit() before predict().")
        X = np.asarray(X, dtype=float)
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        jll = self._joint_log_likelihood(X)
        # log-sum-exp para normalizar
        log_prob_x = np.log(np.sum(np.exp(jll - jll.max(axis=1, keepdims=True)), axis=1))
        log_prob_x += jll.max(axis=1)
        return np.exp(jll - log_prob_x[:, np.newaxis])

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
iris = datasets.load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

nb = GaussianNaiveBayes()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot()
plt.title("Gaussian Naive Bayes - Iris")
plt.show()